Notebook utilizado para exploração das estrategias de feature engineering, enconding e organização dos dados para treinamento dos modelos de machine learning. No final, será obtido um pipeline do Sklearn que consegue transformar cada uma das colunas existentes na base dados para treinamento dos modelos.

In [2]:
import pandas as pd
import sqlite3

KEY_COLS = ['V010100', 'NUM_QUADRA', 'NUM_FACE', 'V010800']

# Agroindústria

Não apresenta nenhum tipo de preprocessamento, pois nenhuma anomalia foi anotada para os dados presentes nessa tabela.

In [3]:
with sqlite3.connect('../data/amstr_geral_criticas.db') as conn:
    df_agroind = pd.read_sql_query('SELECT * FROM agro_ind', conn)

df_agroind.head()

,id,V010100,NUM_QUADRA,NUM_FACE,V010800,V41010200,V41030300,V41030500,V41030600,V41030601,V41030700,VW41030700,VW41030300,VW41030500,VW41030301,VW41030501,VW41030100
0,0,431500805000011,0,98,78,609,5.0,NaN,059,1.0,NaN,NaN,5.0,NaN,NaN,NaN,None
1,1,431500805000011,0,98,78,613,20.0,NaN,059,1.0,NaN,NaN,20.0,NaN,NaN,NaN,None
2,2,431500805000011,0,98,78,616,20.0,NaN,064,1.0,NaN,NaN,20.0,NaN,NaN,NaN,None
3,3,431500805000011,0,98,78,618,60.0,NaN,059,1.0,NaN,NaN,60.0,NaN,NaN,NaN,None
4,4,431500805000011,0,98,78,620,20.0,NaN,059,1.0,NaN,NaN,20.0,NaN,NaN,NaN,None


In [9]:
df_agroind.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28099 entries, 0 to 28098
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          28099 non-null  int64  
 1   V010100     28099 non-null  int64  
 2   NUM_QUADRA  28099 non-null  int64  
 3   NUM_FACE    28099 non-null  int64  
 4   V010800     28099 non-null  int64  
 5   V41010200   28099 non-null  object 
 6   V41030300   28099 non-null  float64
 7   V41030500   11804 non-null  float64
 8   V41030600   28099 non-null  object 
 9   V41030601   28094 non-null  float64
 10  V41030700   9928 non-null   float64
 11  VW41030700  9923 non-null   float64
 12  VW41030300  28094 non-null  float64
 13  VW41030500  11799 non-null  float64
 14  VW41030301  9928 non-null   float64
 15  VW41030501  9928 non-null   float64
 16  VW41030100  0 non-null      object 
dtypes: float64(9), int64(5), object(3)
memory usage: 3.6+ MB


In [10]:
# Compila as análises anteriores (dtype, nulos, cardinalidade e skew) em um único dataframe
cols_analise = [col for col in df_agroind.columns[1:] if col not in KEY_COLS]

df_agroind_resumo = pd.concat(
    [
        df_agroind[cols_analise].dtypes.rename("dtype"),
        df_agroind[cols_analise].isnull().mean().rename("pct_nulos"),
        df_agroind[cols_analise].nunique().rename("cardinalidade"),
        df_agroind[cols_analise].skew().rename("skew"),
    ],
    axis=1,
).sort_values(by="pct_nulos", ascending=False)

print(df_agroind_resumo.to_markdown())

|            | dtype   |   pct_nulos |   cardinalidade |      skew |
|:-----------|:--------|------------:|----------------:|----------:|
| VW41030100 | object  | 1           |               0 | nan       |
| VW41030700 | float64 | 0.646856    |             439 |  31.5512  |
| V41030700  | float64 | 0.646678    |             274 |  43.5626  |
| VW41030301 | float64 | 0.646678    |            1511 |  52.5034  |
| VW41030501 | float64 | 0.646678    |            1544 |  54.5095  |
| VW41030500 | float64 | 0.580092    |            1058 |  68.7561  |
| V41030500  | float64 | 0.579914    |             778 |  78.9678  |
| VW41030300 | float64 | 0.000177942 |            1281 | 149.229   |
| V41030601  | float64 | 0.000177942 |             157 |  79.5474  |
| V41030600  | object  | 0           |              49 |  -0.26392 |
| V41010200  | object  | 0           |              33 |  -0.43692 |
| V41030300  | float64 | 0           |             896 |  81.2848  |


A variável `VW41030300` apresenta um skew muito elevado. Precisamos entender que informação ela armazena e os valores existentes para propormos um fluxo de preprocessamento adequado.

In [12]:
print(df_agroind['VW41030300'].describe())
print(df_agroind['VW41030300'].value_counts().head(20))
print(df_agroind['VW41030300'].quantile([0.90, 0.95, 0.99, 0.999, 1.0]))

count    2.809400e+04
mean     3.905418e+04
std      3.176473e+06
min      0.000000e+00
25%      7.000000e+01
50%      2.000000e+02
75%      6.720000e+02
max      5.100000e+08
Name: VW41030300, dtype: float64
VW41030300
300.0     1317
200.0     1316
100.0     1274
50.0      1100
150.0      969
30.0       760
60.0       758
20.0       750
120.0      681
400.0      647
10.0       594
600.0      585
500.0      563
250.0      500
80.0       480
180.0      478
40.0       468
240.0      397
360.0      377
1000.0     361
Name: count, dtype: int64
0.900    2.500000e+03
0.950    5.500000e+03
0.990    4.000000e+04
0.999    1.954536e+06
1.000    5.100000e+08
Name: VW41030300, dtype: float64


# Pecuária

In [13]:
with sqlite3.connect('../data/amstr_geral_criticas.db') as conn:
    df_pecu = pd.read_sql_query('SELECT * FROM pecu', conn)

df_pecu.head()

,id,V010100,NUM_QUADRA,NUM_FACE,V010800,V14010000,V14010101,V14090302,V14020101,V14040000,...,VW31010200,V32020100,V32030100,V32040000,V32040100,VW29020500,VW14130001,VW14130002,VW14130003,VW22080401
0,0,314050615000006,0,73,73,2,80.0,73.0,1,2,...,NaN,None,None,None,NaN,NaN,None,None,None,None
1,1,521980305000006,0,128,127,None,NaN,NaN,None,None,...,NaN,None,None,None,NaN,NaN,None,None,None,None
2,2,110003105000006,0,116,116,2,200.0,NaN,1,2,...,NaN,None,None,None,NaN,NaN,None,None,None,None
3,3,120005405000009,0,5,5,2,90.0,50.0,1,2,...,NaN,None,None,None,NaN,NaN,None,None,None,None
4,4,312080505000020,0,79,79,2,72.0,53.0,1,2,...,NaN,None,None,None,NaN,NaN,None,None,None,None


In [14]:
df_pecu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91382 entries, 0 to 91381
Columns: 252 entries, id to VW22080401
dtypes: float64(152), int64(5), object(95)
memory usage: 175.7+ MB


In [15]:
# Compila as análises anteriores (dtype, nulos, cardinalidade e skew) em um único dataframe
cols_analise = [col for col in df_pecu.columns[1:] if col not in KEY_COLS]

df_pecu_resumo = pd.concat(
    [
        df_pecu[cols_analise].dtypes.rename("dtype"),
        df_pecu[cols_analise].isnull().mean().rename("pct_nulos"),
        df_pecu[cols_analise].nunique().rename("cardinalidade"),
        df_pecu[cols_analise].skew(numeric_only=True).rename("skew"),
    ],
    axis=1,
).sort_values(by="pct_nulos", ascending=False)

print(df_pecu_resumo.to_markdown())

|            | dtype   |   pct_nulos |   cardinalidade |       skew |
|:-----------|:--------|------------:|----------------:|-----------:|
| VW16010100 | object  |    1        |               0 | nan        |
| VW15010100 | object  |    1        |               0 | nan        |
| VW14010103 | object  |    1        |               0 | nan        |
| VW19010000 | object  |    1        |               0 | nan        |
| VW19010900 | object  |    1        |               0 | nan        |
| VW18101000 | object  |    1        |               0 | nan        |
| VW17101000 | object  |    1        |               0 | nan        |
| VW20010000 | object  |    1        |               0 | nan        |
| VW21010000 | object  |    1        |               0 | nan        |
| VW14130003 | object  |    1        |               0 | nan        |
| VW22080401 | object  |    1        |               0 | nan        |
| V29080202  | object  |    1        |               0 | nan        |
| VW29080201 | objec

Diversas variáveis numéricas apresentam skew muito elevado (>50). As mais críticas são `V14160101` (skew ≈ 175), `V14090302` (skew ≈ 137) e `VW14060000` (skew ≈ 130). Investigamos a seguir as colunas com maior assimetria para entender os valores extremos.

In [16]:
num_cols = df_pecu.select_dtypes(include='number').columns.tolist()
high_skew_cols = df_pecu[num_cols].skew().sort_values(ascending=False)
high_skew_cols = high_skew_cols[high_skew_cols > 50].index.tolist()

for col in high_skew_cols:
    print(f"\n=== {col} (skew={df_pecu[col].skew():.1f}) ===")
    print(df_pecu[col].describe())
    print(df_pecu[col].quantile([0.90, 0.95, 0.99, 0.999, 1.0]))


=== V14160101 (skew=175.3) ===
count    35675.000000
mean        16.262929
std        117.196965
min          0.000000
25%          4.000000
50%         10.000000
75%         20.000000
max      21600.000000
Name: V14160101, dtype: float64
0.900       35.0
0.950       50.0
0.990      100.0
0.999      270.0
1.000    21600.0
Name: V14160101, dtype: float64

=== V14090302 (skew=137.4) ===
count    65321.000000
mean        59.895057
std        350.944917
min          0.000000
25%          8.000000
50%         20.000000
75%         48.000000
max      72149.000000
Name: V14090302, dtype: float64
0.900      110.0
0.950      200.0
0.990      700.0
0.999     2500.0
1.000    72149.0
Name: V14090302, dtype: float64

=== VW14060000 (skew=129.6) ===
count    75588.000000
mean         2.050853
std         12.066315
min          0.000000
25%          0.652000
50%          1.185000
75%          2.000000
max       2372.000000
Name: VW14060000, dtype: float64
0.900       3.333
0.950       5.217
0.990   